In [2]:
import sys
import os




project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb

import pandas as pd
import track_builder as tb

import os
from dotenv import load_dotenv

load_dotenv()

saving_path = os.getenv("SAVING")
output_path = os.path.join(saving_path, "tracks")
os.makedirs(output_path, exist_ok=True)

BASE_PATH = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR      = 2019

MONTHS_TO_LOAD = [1, 2, 3]

USECOLS   = "default"
SAMPLING  = [0, -1]

COLS_REQUIRED = [
    "shipid",
    "date_time_utc",
    "latitude",
    "longitude",
    "astd_cat",
    "flagname"
]


In [3]:


df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=None, remove_nan_rows=COLS_REQUIRED
)





c:\Users\lamin\miniconda3\envs\torch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading ASTD CSVs: 100%|██████████| 3/3 [03:28<00:00, 69.38s/it]


In [9]:
output_path_df = os.path.join(saving_path, "dataframes")

df.to_parquet(output_path_df + "3month_astd_data.parquet", index=False)

In [4]:
# verify if .parquet file exists, if yes load it, else build tracks and save it
parquet_file_path = os.path.join(output_path, "tracks_2019_Q1.parquet")

if os.path.exists(parquet_file_path):
    tracks = pd.read_parquet(parquet_file_path)
else:
    tracks = tb.build_ship_tracks(df,
                                  max_time_gap_hours=25,
                                  max_distance_km=400,
                                  min_track_length=1,
                                  matching_strategy="balanced",
                                  )
    tracks.to_parquet(parquet_file_path)

tracks

,month,segment_id,track_id
0,2019-01,1645,1
1,2019-01,162,2
2,2019-01,3619,3
3,2019-01,2490,4
4,2019-01,2778,5
...,...,...,...
13241,2019-03,20532,12731
13242,2019-03,20626,12732
13243,2019-03,20619,12733
13244,2019-03,20546,12734


In [5]:
tb.get_track_statistics(tracks, df)

{'n_tracks': 12735,
 'n_segments': 13246,
 'avg_length': 1.0401256380054966,
 'max_length': 3,
 'lengths': track_id
 1        1
 2        1
 3        1
 4        1
 5        1
         ..
 12731    1
 12732    1
 12733    1
 12734    1
 12735    1
 Length: 12735, dtype: int64,
 'by_month': month
 2019-01    6941
 2019-02    3830
 2019-03    2475
 dtype: int64,
 'by_ship_type': astd_cat
 bulk carriers                     2390
 fishing vessels                   2254
 general cargo ships               1665
 other activities                  1481
 container ships                   1310
 chemical tankers                   775
 unknown                            637
 passenger ships                    603
 ro-ro cargo ships                  501
 crude oil tankers                  417
 offshore supply ships              355
 gas tankers                        233
 refrigerated cargo ships           216
 oil product tankers                179
 cruise ships                       126
 other serv

In [6]:
# tracks that span at least 2 months
month_counts = tracks.groupby("track_id")["month"].nunique()
multi_month_ids = month_counts[month_counts >= 2].index

print("Number of tracks spanning at least 2 months:", len(multi_month_ids))
tracks[tracks["track_id"].isin(multi_month_ids)].head(10)


Number of tracks spanning at least 2 months: 449


,month,segment_id,track_id
19,2019-01,1541,20
20,2019-02,1418,20
27,2019-01,171,27
28,2019-02,161,27
29,2019-03,262,27
37,2019-01,9019,35
38,2019-02,5670,35
40,2019-01,5901,37
41,2019-02,4573,37
60,2019-01,693,56


In [7]:
# tracks that span at least 3 months
month_counts = tracks.groupby("track_id")["month"].nunique()
multi_month_ids = month_counts[month_counts >= 3].index

print("Number of tracks spanning at least 3 months:", len(multi_month_ids))
tracks[tracks["track_id"].isin(multi_month_ids)].head(10)


Number of tracks spanning at least 3 months: 62


,month,segment_id,track_id
27,2019-01,171,27
28,2019-02,161,27
29,2019-03,262,27
60,2019-01,693,56
61,2019-02,708,56
62,2019-03,740,56
69,2019-01,660,63
70,2019-02,672,63
71,2019-03,703,63
87,2019-01,4490,77


In [ ]:

work = tb.build_light_multi_track_positions(
    track_table=tracks,         
    track_sampling=20,
    positions_df=df,     
    n_tracks_length=2,     
    base_path=BASE_PATH,
    chunksize=500_000,
    progress=True,
    point_stride=10,
    random_state=42,
)

fig = tb.plot_ship_tracks(
    work,
    color_by="track_id",
    color_mode="categorical",
    show_points=True,
    map_style="open-street-map",
    title="Tracks (light multi-track sample)",
)
fig.update_layout(showlegend=False)
fig.show()

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:436: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [ ]:
df_track_76 = tb.load_positions_for_track(track_id=76, track_table=tracks, base_path=BASE_PATH, chunksize=100000)


Loading positions for track 76: 100%|██████████| 3/3 [01:33<00:00, 31.33s/it]


In [14]:
track = 76
fig_test = tb.plot_individual_track(
                track,
                tracks,
                df_track_76,
                show_segments=True,
                map_style="open-street-map",
                title=f"Track {track} ",
            )

fig_test.show()
# fig_test.write_html("fig_58.html", include_plotlyjs='cdn')

In [13]:
df_track_6348 = tb.load_positions_for_track(track_id=6348, track_table=tracks, base_path=BASE_PATH, chunksize=500000)
track = 6348
fig_test = tb.plot_individual_track(
                track,
                tracks,
                df_track_6348,
                show_segments=True,
                map_style="open-street-map",
                title=f"Track {track} ",
            )

fig_test.show()

Loading positions for track 6348: 100%|██████████| 2/2 [01:19<00:00, 39.90s/it]
